In [1]:
import rasterio
from pathlib import Path
import pandas as pd
from dist_s1_enumerator import enumerate_one_dist_s1_product
from dist_s1.data_models.data_utils import get_max_pre_imgs_per_burst_mw

/Users/cmarshak/miniforge3/envs/dist-s1-env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
products = list(Path('sample_prods/OPERA_L3_DIST-ALERT-S1_T11SKU_20250502T015853Z_20251120T201242Z_S1A_30_v0.1/').glob('*.tif'))
sample_layer = products[-1]
sample_layer

PosixPath('sample_prods/OPERA_L3_DIST-ALERT-S1_T11SKU_20250502T015853Z_20251120T201242Z_S1A_30_v0.1/OPERA_L3_DIST-ALERT-S1_T11SKU_20250502T015853Z_20251120T201242Z_S1A_30_v0.1_GEN-METRIC-MAX.tif')

In [3]:
def get_burst_id(opera_rtc_id: str) -> str:
    return opera_rtc_id.split('_')[3]

def get_track_number(burst_id: str) -> int:
    return int(burst_id.split('-')[0][1:])

def get_acq_time(opera_rtc_id: str) -> pd.Timestamp:
    return pd.Timestamp(opera_rtc_id.split('_')[4])

def get_rtc_input_data(layer_path: str | Path) -> dict:
    with rasterio.open(layer_path) as ds:
        tags = ds.tags()

    rtc_inputs = {
        'post_rtc_opera_ids': tags['post_rtc_opera_ids'].split(','),
        'pre_rtc_opera_ids': tags['pre_rtc_opera_ids'].split(','),
        'mgrs_tile_id': tags['mgrs_tile_id'],  
    }
    
    return rtc_inputs

def format_rtc_input_data(rtc_data: dict) -> pd.DataFrame:
    n_pre = len(rtc_data['pre_rtc_opera_ids'])
    df_pre = pd.DataFrame({'opera_id': rtc_data['pre_rtc_opera_ids'],
                           'input_category': ['pre'] * n_pre})
    n_post = len(rtc_data['post_rtc_opera_ids'])
    df_post = pd.DataFrame({'opera_id': rtc_data['post_rtc_opera_ids'],
                           'input_category': ['post'] * n_post})
    df = pd.concat([df_pre, df_post])
    df['opera_id_trunc'] = df.opera_id.map(lambda opera_id: '_'.join(opera_id.split('_')[:5]))

    df['jpl_burst_id'] = df.opera_id.map(get_burst_id)
    df['track_number'] = df.jpl_burst_id.map(get_track_number)
    df['mgrs_tile_id'] = rtc_data['mgrs_tile_id']
    df['acq_dt'] = df.opera_id.map(get_acq_time)
    return df

In [4]:
data = get_rtc_input_data(sample_layer)
df = format_rtc_input_data(data)
df.head()

,opera_id,input_category,opera_id_trunc,jpl_burst_id,track_number,mgrs_tile_id,acq_dt
0,OPERA_L2_RTC-S1_T137-292317-IW2_20240201T01590...,pre,OPERA_L2_RTC-S1_T137-292317-IW2_20240201T015900Z,T137-292317-IW2,137,11SKU,2024-02-01 01:59:00+00:00
1,OPERA_L2_RTC-S1_T137-292317-IW2_20240213T01585...,pre,OPERA_L2_RTC-S1_T137-292317-IW2_20240213T015859Z,T137-292317-IW2,137,11SKU,2024-02-13 01:58:59+00:00
2,OPERA_L2_RTC-S1_T137-292317-IW2_20240225T01585...,pre,OPERA_L2_RTC-S1_T137-292317-IW2_20240225T015859Z,T137-292317-IW2,137,11SKU,2024-02-25 01:58:59+00:00
3,OPERA_L2_RTC-S1_T137-292317-IW2_20240308T01585...,pre,OPERA_L2_RTC-S1_T137-292317-IW2_20240308T015859Z,T137-292317-IW2,137,11SKU,2024-03-08 01:58:59+00:00
4,OPERA_L2_RTC-S1_T137-292317-IW2_20240320T01590...,pre,OPERA_L2_RTC-S1_T137-292317-IW2_20240320T015900Z,T137-292317-IW2,137,11SKU,2024-03-20 01:59:00+00:00


In [5]:
track_numbers = df.track_number.unique().tolist()
if len(track_numbers) > 1:
    if abs(track_numbers[0] - track_numbers[1]) > 1:
        print(f'too many track numbers present: {track_numbers}')

In [6]:
post_ind = df.input_category == 'post'
df_post = df[post_ind].reset_index(drop=True)

pre_ind = df.input_category == 'pre'
df_pre = df[pre_ind].reset_index(drop=True)

In [7]:
post_time_delta = df_post.acq_dt.max() - df_post.acq_dt.min()
if (post_time_delta).days > 1:
    print(f'post-dates span too long: {post_time_delta}')
post_time_delta

Timedelta('0 days 00:00:21')

In [8]:
max_pre_imgs_per_burst = get_max_pre_imgs_per_burst_mw(20, 3)
max_pre_imgs_per_burst

(6, 6, 8)

In [9]:
str(df_post.acq_dt.min().date())

'2025-05-02'

In [10]:
track_numbers[0]

137

In [11]:
df_product_expected = enumerate_one_dist_s1_product(
        df.mgrs_tile_id.iloc[0],
        track_number=track_numbers[0],
        post_date=str(df_post.acq_dt.min().date()),
        lookback_strategy='multi_window',
        delta_lookback_days=(1095, 730, 365),
        max_pre_imgs_per_burst=max_pre_imgs_per_burst
    )

Searching for post-images for track 137 in MGRS tile 11SKU
Searching for pre-images for multi_window baseline
Lookback days (1095, 730, 365) and window days 365 with max pre-images per burst (6, 6, 8)


Windows: 100%|████████████████████| 3/3 [00:26<00:00,  8.68s/it]


In [12]:
df_product_expected['opera_id_trunc'] = df_product_expected.opera_id.map(lambda opera_id: '_'.join(opera_id.split('_')[:5]))

df_product_expected.shape, df.shape

((483, 15), (483, 7))

In [13]:
df_product_expected.head()

,opera_id,jpl_burst_id,acq_dt,acq_date_for_mgrs_pass,polarizations,track_number,pass_id,url_crosspol,url_copol,geometry,mgrs_tile_id,acq_group_id_within_mgrs_tile,track_token,input_category,opera_id_trunc
0,OPERA_L2_RTC-S1_T137-292317-IW2_20220223T01584...,T137-292317-IW2,2022-02-23 01:58:49+00:00,2022-02-23,VV+VH,137,495,https://cumulus.asf.earthdatacloud.nasa.gov/OP...,https://cumulus.asf.earthdatacloud.nasa.gov/OP...,"POLYGON ((-119.96691 33.91905, -118.98085 34.0...",11SKU,4,137,pre,OPERA_L2_RTC-S1_T137-292317-IW2_20220223T015849Z
1,OPERA_L2_RTC-S1_T137-292317-IW2_20220307T01584...,T137-292317-IW2,2022-03-07 01:58:48+00:00,2022-03-07,VV+VH,137,497,https://cumulus.asf.earthdatacloud.nasa.gov/OP...,https://cumulus.asf.earthdatacloud.nasa.gov/OP...,"POLYGON ((-119.96656 33.91929, -118.98049 34.0...",11SKU,4,137,pre,OPERA_L2_RTC-S1_T137-292317-IW2_20220307T015848Z
2,OPERA_L2_RTC-S1_T137-292317-IW2_20220319T01584...,T137-292317-IW2,2022-03-19 01:58:48+00:00,2022-03-19,VV+VH,137,499,https://cumulus.asf.earthdatacloud.nasa.gov/OP...,https://cumulus.asf.earthdatacloud.nasa.gov/OP...,"POLYGON ((-119.96617 33.91926, -118.9801 34.06...",11SKU,4,137,pre,OPERA_L2_RTC-S1_T137-292317-IW2_20220319T015848Z
3,OPERA_L2_RTC-S1_T137-292317-IW2_20220331T01584...,T137-292317-IW2,2022-03-31 01:58:49+00:00,2022-03-31,VV+VH,137,501,https://cumulus.asf.earthdatacloud.nasa.gov/OP...,https://cumulus.asf.earthdatacloud.nasa.gov/OP...,"POLYGON ((-119.9651 33.91915, -118.97906 34.06...",11SKU,4,137,pre,OPERA_L2_RTC-S1_T137-292317-IW2_20220331T015849Z
4,OPERA_L2_RTC-S1_T137-292317-IW2_20220412T01584...,T137-292317-IW2,2022-04-12 01:58:49+00:00,2022-04-12,VV+VH,137,503,https://cumulus.asf.earthdatacloud.nasa.gov/OP...,https://cumulus.asf.earthdatacloud.nasa.gov/OP...,"POLYGON ((-119.96481 33.91856, -118.92299 34.0...",11SKU,4,137,pre,OPERA_L2_RTC-S1_T137-292317-IW2_20220412T015849Z


In [14]:
post_ind = df_product_expected.input_category == 'post'
df_product_expected_post = df_product_expected[post_ind].reset_index(drop=True)

pre_ind = df_product_expected.input_category == 'pre'
df_product_expected_pre = df_product_expected[pre_ind].reset_index(drop=True)

In [15]:
sorted(df_product_expected_post.acq_date_for_mgrs_pass.unique().tolist())

['2025-05-02']

In [16]:
list(reversed(sorted(df_product_expected_pre.acq_date_for_mgrs_pass.unique().tolist())))

['2024-04-25',
 '2024-04-13',
 '2024-04-01',
 '2024-03-20',
 '2024-03-08',
 '2024-02-25',
 '2024-02-13',
 '2024-02-01',
 '2023-05-01',
 '2023-04-19',
 '2023-04-07',
 '2023-03-26',
 '2023-03-14',
 '2023-03-02',
 '2022-04-24',
 '2022-04-12',
 '2022-03-31',
 '2022-03-19',
 '2022-03-07',
 '2022-02-23']

## Burst ids

In [17]:
burst_id_expected = sorted(df_product_expected.jpl_burst_id.unique().tolist())
bust_id_prod = sorted(df.jpl_burst_id.unique().tolist())

In [18]:
burst_id_expected == bust_id_prod

True

In [19]:
len(burst_id_expected)

23

In [20]:
sorted(burst_id_expected)

['T137-292317-IW2',
 'T137-292317-IW3',
 'T137-292318-IW1',
 'T137-292318-IW2',
 'T137-292318-IW3',
 'T137-292319-IW1',
 'T137-292319-IW2',
 'T137-292319-IW3',
 'T137-292320-IW1',
 'T137-292320-IW2',
 'T137-292320-IW3',
 'T137-292321-IW1',
 'T137-292321-IW2',
 'T137-292321-IW3',
 'T137-292322-IW1',
 'T137-292322-IW2',
 'T137-292322-IW3',
 'T137-292323-IW1',
 'T137-292323-IW2',
 'T137-292323-IW3',
 'T137-292324-IW1',
 'T137-292324-IW2',
 'T137-292325-IW1']

## Product ids

In [21]:
pre_rtc_ids_expected_but_not_found = [rtc_id for rtc_id in df_product_expected_pre.opera_id_trunc.tolist()
                                              if rtc_id not in df_pre.opera_id_trunc.tolist()]
pre_rtc_ids_expected_but_not_found

[]

In [22]:
pre_found_but_not_expected = [rtc_id for rtc_id in df_pre.opera_id_trunc.tolist()
                               if rtc_id not in df_product_expected_pre.opera_id_trunc.tolist()]
pre_found_but_not_expected

[]

In [23]:
post_rtc_ids_expected_but_not_found = [rtc_id for rtc_id in df_product_expected_post.opera_id_trunc.tolist()
                                              if rtc_id not in df_post.opera_id_trunc.tolist()]
post_rtc_ids_expected_but_not_found

[]

In [24]:
post_found_but_not_expected = [rtc_id for rtc_id in df_post.opera_id_trunc.tolist()
                               if rtc_id not in df_product_expected_post.opera_id_trunc.tolist()]
post_found_but_not_expected

[]

# Histogram

In [25]:
from typing import Dict, Tuple, Any

def count_acquisitions_by_window(
    df_prod: pd.DataFrame, ref_date: Any
) -> Dict[str, Tuple[int, int, int]]:

    df_prod['days_ago'] = (ref_date - df_prod['acq_dt']).dt.days

    # 3. Define the bins for 'days_ago'
    # Bins: (365, 730], (730, 1095], (1095, 1460]
    bins = [365, 730, 1095, 1500]
    labels = ['W1', 'W2', 'W3'] 

    # Use pd.cut to assign each row to a time window
    df_prod['time_window'] = pd.cut(df_prod['days_ago'], bins=bins, labels=labels, right=False)

    # 4. Group by 'burst_id' and 'time_window', then unstack to get the counts
    counts_df = (
        df_prod.dropna(subset=['time_window'])
        .groupby(['jpl_burst_id', 'time_window'])
        .size()
        .unstack(fill_value=0)
    )

    # 5. Ensure all 3 window columns exist and are in the correct order
    for label in labels:
        if label not in counts_df.columns:
            counts_df[label] = 0

    counts_df = counts_df[labels]

    # 6. Convert the resulting DataFrame to the desired dictionary format
    result_dict = counts_df.apply(tuple, axis=1).to_dict()

    return result_dict


def get_ordered_dates_by_burst(df: pd.DataFrame) -> dict:
    df_copy = df.copy()
    def sort_and_format_dates(series: pd.Series) -> list[str]:
        sorted_dates = series.sort_values(ascending=False)
        return sorted_dates.dt.strftime('%Y-%m-%d').tolist()

    ordered_dates = (
        df_copy.groupby('jpl_burst_id')['acq_dt']
        .apply(sort_and_format_dates)
        .to_dict()
    )

    return ordered_dates

In [26]:
count_acquisitions_by_window(df_pre, df_product_expected_post.acq_dt.min())

/var/folders/0p/d5x2m4tx5kg1246bplsvyfyh0000gq/T/ipykernel_91493/2362134857.py:20: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(['jpl_burst_id', 'time_window'])


{'T137-292317-IW2': (8, 6, 6),
 'T137-292317-IW3': (8, 6, 6),
 'T137-292318-IW1': (8, 6, 6),
 'T137-292318-IW2': (8, 6, 6),
 'T137-292318-IW3': (8, 6, 6),
 'T137-292319-IW1': (8, 6, 6),
 'T137-292319-IW2': (8, 6, 6),
 'T137-292319-IW3': (8, 6, 6),
 'T137-292320-IW1': (8, 6, 6),
 'T137-292320-IW2': (8, 6, 6),
 'T137-292320-IW3': (8, 6, 6),
 'T137-292321-IW1': (8, 6, 6),
 'T137-292321-IW2': (8, 6, 6),
 'T137-292321-IW3': (8, 6, 6),
 'T137-292322-IW1': (8, 6, 6),
 'T137-292322-IW2': (8, 6, 6),
 'T137-292322-IW3': (8, 6, 6),
 'T137-292323-IW1': (8, 6, 6),
 'T137-292323-IW2': (8, 6, 6),
 'T137-292323-IW3': (8, 6, 6),
 'T137-292324-IW1': (8, 6, 6),
 'T137-292324-IW2': (8, 6, 6),
 'T137-292325-IW1': (8, 6, 6)}

In [27]:
count_acquisitions_by_window(df_product_expected_pre, df_product_expected_post.acq_dt.min())

/var/folders/0p/d5x2m4tx5kg1246bplsvyfyh0000gq/T/ipykernel_91493/2362134857.py:20: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(['jpl_burst_id', 'time_window'])


{'T137-292317-IW2': (8, 6, 6),
 'T137-292317-IW3': (8, 6, 6),
 'T137-292318-IW1': (8, 6, 6),
 'T137-292318-IW2': (8, 6, 6),
 'T137-292318-IW3': (8, 6, 6),
 'T137-292319-IW1': (8, 6, 6),
 'T137-292319-IW2': (8, 6, 6),
 'T137-292319-IW3': (8, 6, 6),
 'T137-292320-IW1': (8, 6, 6),
 'T137-292320-IW2': (8, 6, 6),
 'T137-292320-IW3': (8, 6, 6),
 'T137-292321-IW1': (8, 6, 6),
 'T137-292321-IW2': (8, 6, 6),
 'T137-292321-IW3': (8, 6, 6),
 'T137-292322-IW1': (8, 6, 6),
 'T137-292322-IW2': (8, 6, 6),
 'T137-292322-IW3': (8, 6, 6),
 'T137-292323-IW1': (8, 6, 6),
 'T137-292323-IW2': (8, 6, 6),
 'T137-292323-IW3': (8, 6, 6),
 'T137-292324-IW1': (8, 6, 6),
 'T137-292324-IW2': (8, 6, 6),
 'T137-292325-IW1': (8, 6, 6)}

In [28]:
pd.Timestamp('2023-12-10') - pd.Timestamp('2021-12-14')

Timedelta('726 days 00:00:00')

In [29]:
print(df_product_expected_post.acq_dt.max())
get_ordered_dates_by_burst(df_product_expected_pre)

2025-05-02 01:59:14+00:00


{'T137-292317-IW2': ['2024-04-25',
  '2024-04-13',
  '2024-04-01',
  '2024-03-20',
  '2024-03-08',
  '2024-02-25',
  '2024-02-13',
  '2024-02-01',
  '2023-05-01',
  '2023-04-19',
  '2023-04-07',
  '2023-03-26',
  '2023-03-14',
  '2023-03-02',
  '2022-04-24',
  '2022-04-12',
  '2022-03-31',
  '2022-03-19',
  '2022-03-07',
  '2022-02-23'],
 'T137-292317-IW3': ['2024-04-25',
  '2024-04-13',
  '2024-04-01',
  '2024-03-20',
  '2024-03-08',
  '2024-02-25',
  '2024-02-13',
  '2024-02-01',
  '2023-05-01',
  '2023-04-19',
  '2023-04-07',
  '2023-03-26',
  '2023-03-14',
  '2023-03-02',
  '2022-04-24',
  '2022-04-12',
  '2022-03-31',
  '2022-03-19',
  '2022-03-07',
  '2022-02-23'],
 'T137-292318-IW1': ['2024-04-25',
  '2024-04-13',
  '2024-04-01',
  '2024-03-20',
  '2024-03-08',
  '2024-02-25',
  '2024-02-13',
  '2024-02-01',
  '2023-05-01',
  '2023-04-19',
  '2023-04-07',
  '2023-03-26',
  '2023-03-14',
  '2023-03-02',
  '2022-04-24',
  '2022-04-12',
  '2022-03-31',
  '2022-03-19',
  '2022-03-07

In [30]:
count_acquisitions_by_window(df_pre, df_product_expected_post.acq_dt.min())

/var/folders/0p/d5x2m4tx5kg1246bplsvyfyh0000gq/T/ipykernel_91493/2362134857.py:20: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(['jpl_burst_id', 'time_window'])


{'T137-292317-IW2': (8, 6, 6),
 'T137-292317-IW3': (8, 6, 6),
 'T137-292318-IW1': (8, 6, 6),
 'T137-292318-IW2': (8, 6, 6),
 'T137-292318-IW3': (8, 6, 6),
 'T137-292319-IW1': (8, 6, 6),
 'T137-292319-IW2': (8, 6, 6),
 'T137-292319-IW3': (8, 6, 6),
 'T137-292320-IW1': (8, 6, 6),
 'T137-292320-IW2': (8, 6, 6),
 'T137-292320-IW3': (8, 6, 6),
 'T137-292321-IW1': (8, 6, 6),
 'T137-292321-IW2': (8, 6, 6),
 'T137-292321-IW3': (8, 6, 6),
 'T137-292322-IW1': (8, 6, 6),
 'T137-292322-IW2': (8, 6, 6),
 'T137-292322-IW3': (8, 6, 6),
 'T137-292323-IW1': (8, 6, 6),
 'T137-292323-IW2': (8, 6, 6),
 'T137-292323-IW3': (8, 6, 6),
 'T137-292324-IW1': (8, 6, 6),
 'T137-292324-IW2': (8, 6, 6),
 'T137-292325-IW1': (8, 6, 6)}

In [31]:
print(df_product_expected_post.acq_dt.max())
get_ordered_dates_by_burst(df_pre)

2025-05-02 01:59:14+00:00


{'T137-292317-IW2': ['2024-04-25',
  '2024-04-13',
  '2024-04-01',
  '2024-03-20',
  '2024-03-08',
  '2024-02-25',
  '2024-02-13',
  '2024-02-01',
  '2023-05-01',
  '2023-04-19',
  '2023-04-07',
  '2023-03-26',
  '2023-03-14',
  '2023-03-02',
  '2022-04-24',
  '2022-04-12',
  '2022-03-31',
  '2022-03-19',
  '2022-03-07',
  '2022-02-23'],
 'T137-292317-IW3': ['2024-04-25',
  '2024-04-13',
  '2024-04-01',
  '2024-03-20',
  '2024-03-08',
  '2024-02-25',
  '2024-02-13',
  '2024-02-01',
  '2023-05-01',
  '2023-04-19',
  '2023-04-07',
  '2023-03-26',
  '2023-03-14',
  '2023-03-02',
  '2022-04-24',
  '2022-04-12',
  '2022-03-31',
  '2022-03-19',
  '2022-03-07',
  '2022-02-23'],
 'T137-292318-IW1': ['2024-04-25',
  '2024-04-13',
  '2024-04-01',
  '2024-03-20',
  '2024-03-08',
  '2024-02-25',
  '2024-02-13',
  '2024-02-01',
  '2023-05-01',
  '2023-04-19',
  '2023-04-07',
  '2023-03-26',
  '2023-03-14',
  '2023-03-02',
  '2022-04-24',
  '2022-04-12',
  '2022-03-31',
  '2022-03-19',
  '2022-03-07

In [32]:
print(df_product_expected_post.acq_dt.max())
get_ordered_dates_by_burst(df_product_expected_pre)

2025-05-02 01:59:14+00:00


{'T137-292317-IW2': ['2024-04-25',
  '2024-04-13',
  '2024-04-01',
  '2024-03-20',
  '2024-03-08',
  '2024-02-25',
  '2024-02-13',
  '2024-02-01',
  '2023-05-01',
  '2023-04-19',
  '2023-04-07',
  '2023-03-26',
  '2023-03-14',
  '2023-03-02',
  '2022-04-24',
  '2022-04-12',
  '2022-03-31',
  '2022-03-19',
  '2022-03-07',
  '2022-02-23'],
 'T137-292317-IW3': ['2024-04-25',
  '2024-04-13',
  '2024-04-01',
  '2024-03-20',
  '2024-03-08',
  '2024-02-25',
  '2024-02-13',
  '2024-02-01',
  '2023-05-01',
  '2023-04-19',
  '2023-04-07',
  '2023-03-26',
  '2023-03-14',
  '2023-03-02',
  '2022-04-24',
  '2022-04-12',
  '2022-03-31',
  '2022-03-19',
  '2022-03-07',
  '2022-02-23'],
 'T137-292318-IW1': ['2024-04-25',
  '2024-04-13',
  '2024-04-01',
  '2024-03-20',
  '2024-03-08',
  '2024-02-25',
  '2024-02-13',
  '2024-02-01',
  '2023-05-01',
  '2023-04-19',
  '2023-04-07',
  '2023-03-26',
  '2023-03-14',
  '2023-03-02',
  '2022-04-24',
  '2022-04-12',
  '2022-03-31',
  '2022-03-19',
  '2022-03-07